# EDA Focus Notebook: Summary and Regression Analysis

This notebook is a focused version of the broader EDA work.
It is centered on two things:

- summary findings about the dataset
- exploratory regression analysis

The goal is to give a team-friendly notebook that is easier to present, revise, and discuss than the full end-to-end script.

## What this notebook covers

1. Load the source data
2. Build the hourly aligned panel used for joint analysis
3. Summarize the most important dataset facts
4. Run single-variable regressions against price
5. Compare weather vs price and weather vs load using R^2
6. Run two-variable regressions against price
7. Give plain-English interpretations of the main regression results

In [ ]:
from pathlib import Path
import sys
from itertools import combinations

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve()
if not (ROOT / "old").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from eda.eda import (
    SOURCE_FILES,
    TARGET,
    WEATHER_COLUMNS,
    PRODUCTION_COLUMNS,
    EXOGENOUS_COLUMNS,
    load_csv,
    summarize_dataset,
    build_dataset_inventory,
    summarize_frequency_transition,
    add_calendar_features,
    data_quality_checks,
    fit_ols_model,
)

TABLE_DIR = ROOT / "eda" / "tables"
FIG_DIR = ROOT / "eda" / "figures" / "static"

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Load the source data and summarize the files

In [ ]:
source_frames = {name: load_csv(path) for name, path in SOURCE_FILES.items()}
source_summaries = [summarize_dataset(name, SOURCE_FILES[name], df) for name, df in source_frames.items()]
price_transition = summarize_frequency_transition(source_frames["price_source"])
inventory_df, notes_df = build_dataset_inventory(source_summaries, [], price_transition["first_non_hourly_timestamp"])

inventory_overview = inventory_df[["dataset", "rows", "date_start", "date_end", "frequency"]].copy()
inventory_overview

In [ ]:
notes_df

## 2. Build the hourly aligned panel

We keep only top-of-hour price observations and then left-join weather, load, and generation.
This gives us one shared panel for summary and regression work.

In [ ]:
hourly_price = source_frames["price_source"][source_frames["price_source"]["time"].dt.minute == 0].copy()
hourly_panel = (
    hourly_price.merge(source_frames["weather_source"], on="time", how="left")
    .merge(source_frames["load_source"], on="time", how="left")
    .merge(source_frames["generation_source"], on="time", how="left")
)
hourly_panel = add_calendar_features(hourly_panel).sort_values("time").set_index("time")

quality_table, _ = data_quality_checks(hourly_panel.reset_index())

summary_table = pd.DataFrame(
    [
        {"item": "Hourly panel rows", "value": len(hourly_panel)},
        {"item": "Panel start", "value": hourly_panel.index.min()},
        {"item": "Panel end", "value": hourly_panel.index.max()},
        {"item": "Negative prices", "value": int((hourly_panel[TARGET] < 0).sum())},
        {"item": "Price min", "value": float(hourly_panel[TARGET].min())},
        {"item": "Price max", "value": float(hourly_panel[TARGET].max())},
        {"item": "First quarter-hour price timestamp", "value": price_transition["first_non_hourly_timestamp"]},
    ]
)
summary_table

In [ ]:
quality_table[["column", "missing_count", "missing_pct", "robust_anomaly_count", "std"]].sort_values(
    ["missing_pct", "robust_anomaly_count"], ascending=False
)

## 3. Single-variable regressions against price

Each model below answers a simple question like:

- load vs price
- temperature vs price
- wind speed vs price

These are **not** final forecasting models.
They are simple screens that help us see whether one variable alone has a meaningful linear relationship with price.

In [ ]:
simple_rows = []
simple_coef_rows = []

for feature in EXOGENOUS_COLUMNS:
    result, fitted_df, coef_df = fit_ols_model(
        hourly_panel.reset_index(),
        TARGET,
        [feature],
        f"Notebook Simple OLS: price ~ {feature}",
        f"notebook_simple_{feature}",
    )
    coef_row = coef_df[coef_df["variable"] == feature].iloc[0]
    simple_rows.append(
        {
            "predictor": feature,
            "coefficient": coef_row["coefficient"],
            "coefficient_p_value": coef_row["p_value"],
            "model_pvalue": result.model_pvalue,
            "r2": result.r2,
            "adj_r2": result.adj_r2,
            "explained_variance": result.explained_variance,
            "nobs": result.nobs,
            "direction": "rises" if coef_row["coefficient"] > 0 else "falls" if coef_row["coefficient"] < 0 else "flat",
        }
    )
    simple_coef_rows.append(coef_df)

simple_metrics = pd.DataFrame(simple_rows).sort_values("adj_r2", ascending=False)
simple_metrics.to_csv(TABLE_DIR / "notebook_price_single_variable_regression_metrics.csv", index=False)
simple_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = simple_metrics.sort_values("adj_r2", ascending=True)
bars = ax.barh(plot_df["predictor"], plot_df["adj_r2"], color="#4c72b0")
ax.set_title("Single-variable price regressions ranked by adjusted R^2")
ax.set_xlabel("Adjusted R^2")
ax.set_ylabel("Predictor")
for bar, (_, row) in zip(bars, plot_df.iterrows()):
    ax.text(
        bar.get_width() + 0.0005,
        bar.get_y() + bar.get_height() / 2,
        f"R^2={row['r2']:.4f} | p={row['model_pvalue']:.3g}",
        va="center",
        fontsize=9,
    )
fig.tight_layout()
fig.savefig(FIG_DIR / "notebook_price_single_variable_regression_ranking.png", dpi=180, bbox_inches="tight")
plt.show()

## 4. Weather vs price and weather vs load using R^2

This section answers:

- which weather variables explain price better?
- which weather variables explain load better?
- is weather more directly tied to load than to price?

In [ ]:
weather_rows = []

for target_col in [TARGET, "total_load"]:
    for feature in WEATHER_COLUMNS:
        result, _, coef_df = fit_ols_model(
            hourly_panel.reset_index(),
            target_col,
            [feature],
            f"Notebook Weather OLS: {target_col} ~ {feature}",
            f"notebook_weather_{target_col}_{feature}",
        )
        coef_row = coef_df[coef_df["variable"] == feature].iloc[0]
        weather_rows.append(
            {
                "target_column": target_col,
                "feature": feature,
                "coefficient": coef_row["coefficient"],
                "model_pvalue": result.model_pvalue,
                "r2": result.r2,
                "adj_r2": result.adj_r2,
                "explained_variance": result.explained_variance,
                "nobs": result.nobs,
            }
        )

weather_metrics = pd.DataFrame(weather_rows)
weather_pivot = weather_metrics.pivot(index="feature", columns="target_column", values="r2").reset_index()
weather_pivot.columns = ["feature", "r2_price", "r2_total_load"]
weather_pivot["load_minus_price"] = weather_pivot["r2_total_load"] - weather_pivot["r2_price"]
weather_pivot.to_csv(TABLE_DIR / "notebook_weather_r2_price_vs_load.csv", index=False)
weather_pivot.sort_values("r2_total_load", ascending=False)

In [ ]:
plot_df = weather_pivot.melt(id_vars="feature", var_name="target", value_name="r2")
plot_df["target"] = plot_df["target"].map({"r2_price": "Price", "r2_total_load": "Total Load"})

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=plot_df, x="feature", y="r2", hue="target", ax=ax)
ax.set_title("Weather explanatory power: price vs total load")
ax.set_xlabel("Weather feature")
ax.set_ylabel("R^2")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(FIG_DIR / "notebook_weather_r2_price_vs_load.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Two-variable regressions against price

This section checks combinations such as:

- cloud cover + wind speed vs price
- load + price helper variables
- weather + generation combinations

This is still exploratory, but it is more realistic than looking at only one variable at a time.

In [ ]:
pair_rows = []

for feature_a, feature_b in combinations(EXOGENOUS_COLUMNS, 2):
    features = [feature_a, feature_b]
    label = " + ".join(features)
    slug = "__".join(features)
    result, _, coef_df = fit_ols_model(
        hourly_panel.reset_index(),
        TARGET,
        features,
        f"Notebook Pairwise OLS: price ~ {label}",
        f"notebook_pair_{slug}",
    )
    coef_subset = coef_df[coef_df["variable"] != "const"].copy()
    feature_effects = " | ".join(
        f"{row.variable}: coef={row.coefficient:.4f}, p={row.p_value:.3g}"
        for row in coef_subset.itertuples()
    )
    pair_rows.append(
        {
            "model_features": label,
            "model_pvalue": result.model_pvalue,
            "r2": result.r2,
            "adj_r2": result.adj_r2,
            "explained_variance": result.explained_variance,
            "nobs": result.nobs,
            "feature_effects": feature_effects,
        }
    )

pairwise_metrics = pd.DataFrame(pair_rows).sort_values("adj_r2", ascending=False)
pairwise_metrics.to_csv(TABLE_DIR / "notebook_price_two_variable_regression_metrics.csv", index=False)
pairwise_metrics.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
plot_df = pairwise_metrics.head(10).sort_values("adj_r2", ascending=True)
bars = ax.barh(plot_df["model_features"], plot_df["adj_r2"], color="#55a868")
ax.set_title("Top two-variable price regressions ranked by adjusted R^2")
ax.set_xlabel("Adjusted R^2")
ax.set_ylabel("Model")
for bar, (_, row) in zip(bars, plot_df.iterrows()):
    ax.text(
        bar.get_width() + 0.0005,
        bar.get_y() + bar.get_height() / 2,
        f"R^2={row['r2']:.4f} | p={row['model_pvalue']:.3g}",
        va="center",
        fontsize=9,
    )
fig.tight_layout()
fig.savefig(FIG_DIR / "notebook_price_two_variable_regression_ranking.png", dpi=180, bbox_inches="tight")
plt.show()

## 6. Plain-English interpretation of selected examples

These examples answer the questions people usually ask first.

In [ ]:
load_row = simple_metrics[simple_metrics["predictor"] == "total_load"].iloc[0]
temp_row = simple_metrics[simple_metrics["predictor"] == "temperature_2m"].iloc[0]
wind_row = simple_metrics[simple_metrics["predictor"] == "wind_speed_10m"].iloc[0]
cloud_wind_row = pairwise_metrics[pairwise_metrics["model_features"] == "cloud_cover + wind_speed_10m"].iloc[0]

interpretation_table = pd.DataFrame(
    [
        {
            "question": "Does price rise when load rises?",
            "short_answer": "No in the simple same-time screen; fitted price falls slightly.",
            "evidence": f"coef={load_row['coefficient']:.6f}, p={load_row['model_pvalue']:.3g}, R^2={load_row['r2']:.4f}",
            "why_it_matters": "The p-value is small, but the explanatory power is tiny.",
        },
        {
            "question": "Does price rise when temperature rises?",
            "short_answer": "No in the simple same-time screen; fitted price falls.",
            "evidence": f"coef={temp_row['coefficient']:.4f}, p={temp_row['model_pvalue']:.3g}, R^2={temp_row['r2']:.4f}",
            "why_it_matters": "The relation is statistically detectable but still very weak.",
        },
        {
            "question": "Does price rise when wind speed rises?",
            "short_answer": "No in the simple same-time screen; fitted price falls.",
            "evidence": f"coef={wind_row['coefficient']:.4f}, p={wind_row['model_pvalue']:.3g}, R^2={wind_row['r2']:.4f}",
            "why_it_matters": "This is the strongest single-variable price screen, but it still explains only a modest share of variance.",
        },
        {
            "question": "What about cloud cover and wind speed together?",
            "short_answer": "The combined model is better than cloud cover alone and slightly better than wind speed alone.",
            "evidence": f"adj_R^2={cloud_wind_row['adj_r2']:.4f}, p={cloud_wind_row['model_pvalue']:.3g}",
            "why_it_matters": cloud_wind_row['feature_effects'],
        },
    ]
)
interpretation_table.to_csv(TABLE_DIR / "notebook_regression_interpretation_examples.csv", index=False)
interpretation_table

## 7. Main takeaways from this focused notebook

- The target is volatile and not easy to explain with one variable at a time.
- Single-variable regressions are useful for screening, not final forecasting.
- Weather is more directly tied to load than to price in these simple linear screens.
- Some p-values are tiny because the dataset is large, even when the actual explanatory power is small.
- Forecasting decisions should not be based on p-value alone. We also need leakage safety, time-aware validation, and operational feature availability.